In [ ]:
import sys
import os
import os
import json
import pandas as pd
from glob import glob
from sklearn.model_selection import train_test_split

sys.path.append("../")

from analysis_utils import prepare_data

RESTRICT_TO_VALID = True

REF_ANN_PATH = os.path.realpath(
    "../color-grid-segment-annotations/data/processed_annotations/consensus_majority_clean.json"
)  # preprocessed annotations for this project
COLORGRID_ANN_PATH = os.path.realpath(
    "../color-grid-segment-annotations/data/color_grid_data.json"
)  # original annotations (formatted)
LS_DATA_PATH = os.path.realpath(
    "../color-grid-segment-annotations/data/label_studio_input/label_studio_input.json"
)  # label studio inputs

SAMPLE_INDEX_FILE = os.path.realpath("../human_eval/sample_ids.json")

In [2]:
# load data
_, data_df, _, _ = prepare_data(REF_ANN_PATH, COLORGRID_ANN_PATH, LS_DATA_PATH)
data_df = data_df.set_index("round_id")

# load sample ids
with open(SAMPLE_INDEX_FILE, "r") as f:
    sample_ids = json.load(f)

read ann data...
read source data...
add further information from label studio data...
calculate further stats...


In [3]:
sample_files = glob('samples/sample_*.json')

In [4]:
sample_dfs = []

for file in sample_files:
    sample_file_df = pd.read_json(file)
    if '_thinking' in file:
        sample_file_df["system"] = sample_file_df.system + '_thinking'
    sample_file_df["description"] = sample_file_df.description.map(lambda x: x.strip() if isinstance(x, str) else x)
    sample_file_df["condition"] = data_df.loc[sample_file_df.target_id].condition.values
    sample_dfs.append(sample_file_df)

In [5]:
# merge all samples
all_samples = pd.concat(sample_dfs)

# make unique item_id (target_id + system) and set as index
all_samples["item_id"] = all_samples["target_id"] + '_' + all_samples["system"]
all_samples = all_samples.set_index("item_id", drop=False)
# create stratification label for splitting
all_samples["stratification_label"] = all_samples["system"] + "_" + all_samples["condition"]

In [6]:
# print the number of samples per system and condition
print(all_samples.groupby(["system", "condition"]).size())
all_samples.sample(5)

system                                            condition
Qwen/Qwen3.5-27B-FP8                              CLOSE        50
                                                  FAR          50
                                                  SPLIT        50
Qwen/Qwen3.5-27B-FP8_thinking                     CLOSE        50
                                                  FAR          50
                                                  SPLIT        50
Qwen/Qwen3.5-4B                                   CLOSE        50
                                                  FAR          50
                                                  SPLIT        50
Qwen/Qwen3.5-4B_thinking                          CLOSE        50
                                                  FAR          50
                                                  SPLIT        50
Qwen/Qwen3.5-9B                                   CLOSE        50
                                                  FAR          50
                

,target_id,description,system,condition,item_id,stratification_label
item_id,,,,,,
5704-eb65850a-01b8-43b7-ab26-937928c2f46c_57_RedHatAI/gemma-4-26B-A4B-it-FP8-Dynamic,5704-eb65850a-01b8-43b7-ab26-937928c2f46c_57,"A grid with gray, yellow, and cyan in the top ...",RedHatAI/gemma-4-26B-A4B-it-FP8-Dynamic,FAR,5704-eb65850a-01b8-43b7-ab26-937928c2f46c_57_R...,RedHatAI/gemma-4-26B-A4B-it-FP8-Dynamic_FAR
1463-5c2c95c7-2401-41ff-b34c-98e93536f94c_46_RedHatAI/gemma-4-26B-A4B-it-FP8-Dynamic,1463-5c2c95c7-2401-41ff-b34c-98e93536f94c_46,"Red, tan, magenta, brown, purple, tan, green, ...",RedHatAI/gemma-4-26B-A4B-it-FP8-Dynamic,SPLIT,1463-5c2c95c7-2401-41ff-b34c-98e93536f94c_46_R...,RedHatAI/gemma-4-26B-A4B-it-FP8-Dynamic_SPLIT
5704-eb65850a-01b8-43b7-ab26-937928c2f46c_1_Qwen/Qwen3.5-27B-FP8_thinking,5704-eb65850a-01b8-43b7-ab26-937928c2f46c_1,The grid with a bottom row consisting of light...,Qwen/Qwen3.5-27B-FP8_thinking,SPLIT,5704-eb65850a-01b8-43b7-ab26-937928c2f46c_1_Qw...,Qwen/Qwen3.5-27B-FP8_thinking_SPLIT
9193-e80b9de9-7035-4132-a05d-786e925f9e1c_39_google/gemma-4-E4B-it,9193-e80b9de9-7035-4132-a05d-786e925f9e1c_39,"Green, gray, and teal with a red and blue row ...",google/gemma-4-E4B-it,FAR,9193-e80b9de9-7035-4132-a05d-786e925f9e1c_39_g...,google/gemma-4-E4B-it_FAR
6113-38555c00-9ed9-463a-ab14-04cdfb258539_13_Qwen/Qwen3.5-9B,6113-38555c00-9ed9-463a-ab14-04cdfb258539_13,"The grid contains a grey square, a green squar...",Qwen/Qwen3.5-9B,SPLIT,6113-38555c00-9ed9-463a-ab14-04cdfb258539_13_Q...,Qwen/Qwen3.5-9B_SPLIT


In [7]:
# create stratified splits for individual annotators

# First split: 1/3 vs 2/3
split1, df_temp = train_test_split(
    all_samples,
    test_size=2/3,
    stratify=all_samples["stratification_label"],
    random_state=42,
)

# Second split: split remaining 2/3 into two equal parts
split2, split3 = train_test_split(
    df_temp,
    test_size=0.5,
    stratify=df_temp["stratification_label"],
    random_state=42,
)

assert pd.concat([split1, split2, split3]).sort_index().equals(all_samples.sort_index())

In [8]:
# create set of shared items across all splits
# to evaluate inter-annotator agreement

def sample_per_label(df, n=1, seed=42):
    return df.groupby("stratification_label", group_keys=False).sample(n=n, random_state=seed)

# sample 1 item per label from each split
s1 = sample_per_label(split1, n=1, seed=42)
s2 = sample_per_label(split2, n=1, seed=42)
s3 = sample_per_label(split3, n=1, seed=42)

# add each sampled set into every split
split1_aug = pd.concat([split1, s2, s3])
split2_aug = pd.concat([split2, s1, s3])
split3_aug = pd.concat([split3, s1, s2])

# sanity check: shared items appear in each split
all_shared = set(split1_aug.index) & set(split2_aug.index) & set(split3_aug.index)
print(len(all_shared), "items are shared across all splits")

assert all_shared.issubset(split1_aug.index)
assert all_shared.issubset(split2_aug.index)
assert all_shared.issubset(split3_aug.index)

# sanity check: splits are disjoint except for shared items
def unexpected_overlap(a, b, shared):
    return (set(a.index) & set(b.index)) - shared

assert len(unexpected_overlap(split1_aug, split2_aug, all_shared)) == 0, "Unexpected overlap between split1 and split2"
assert len(unexpected_overlap(split1_aug, split3_aug, all_shared)) == 0, "Unexpected overlap between split1 and split3"
assert len(unexpected_overlap(split2_aug, split3_aug, all_shared)) == 0, "Unexpected overlap between split2 and split3"

# sanity check: each split has roughly equal number of items per system and condition
# (at most 1, since the number of items per label is not always divisible by 3)
for df in [split1_aug, split2_aug, split3_aug]:
    _df = df.groupby(["system", "condition"]).size()
    assert ((_df.max() - _df.min()) <= 1).all()

135 items are shared across all splits


In [9]:
out_dir = "annotator_splits"
os.makedirs(out_dir, exist_ok=True)

cols_to_keep = ["item_id","target_id","description","system","condition"]

for i, split in enumerate([split1_aug, split2_aug, split3_aug], start=1):
    print(f"Saving split {i} with {len(split)} items to {out_dir}")
    split[cols_to_keep].to_json(os.path.join(out_dir, f"annotator_split_{i}.json"), orient="records", indent=2)

Saving split 1 with 840 items to annotator_splits
Saving split 2 with 840 items to annotator_splits
Saving split 3 with 840 items to annotator_splits
